## Imports

In [16]:
# Python standards
import numpy as np
import pandas as pd 
import seaborn as sns
import os
import csv
import mygene

# Project Specific
import mygene
from Bio import Entrez

## Import Data

In [17]:
# Import data files - be able to differentiate between .csv and txt

# Makes index column (row labels) lowercase and string format
def lowercase_index(table):
    table.index = table.index.map(str)
    table.index = table.index.str.lower()
    return table

# Turn input data into pandas tables
def make_table(filepath):
    filename, extension = os.path.splitext(filepath)

    # Convert .csv file to 
    if extension == ".csv":
        table = pd.read_csv(filepath, header=0, index_col=0) 

    # For non .csv files (.txt, .tsv), use Sniffer to automatically detect delimiter type
    elif extension in [".txt", ".tsv"]:
        with open(filepath, 'r') as f1:
            dialect = csv.Sniffer().sniff(f1.readline())
            delimiter = dialect.delimiter
        table = pd.read_csv(filepath, sep=delimiter, header=0, index_col=0)

    # Raise error for non-supported file types
    else:
        raise TypeError("Unsupported format - file must be .csv, .txt, or .tsv")

    # Convert index (row names) to strings and all lowercase - this makes future processing/matching much easier  
    table = lowercase_index(table)
    table = table.astype(int)
    
    return table

In [18]:
gse167216 = make_table("/home/yaogilbe/CMSE410/cmse410-final-project/GSE167216_Raw_gene_counts_matrix.txt")
gse167216

,I00001,I00002,I00003,I00004,I00005,I00006,I00007,I00008,I00009,I00010,...,I00027,I00028,I00029,I00030,I00031,I00032,I00033,I00034,I00035,I00036
0610007p14rik,858,901,1053,1413,882,704,947,747,1253,928,...,1108,1293,1510,1080,474,1098,857,1187,1572,1111
0610009b22rik,430,515,450,423,395,341,392,449,556,382,...,395,428,515,522,625,487,332,406,440,459
0610009l18rik,14,10,12,9,8,8,12,11,8,5,...,13,7,10,8,10,8,6,7,20,13
0610009o20rik,600,462,559,541,441,268,459,523,594,499,...,475,441,644,479,619,457,437,508,690,378
0610010f05rik,615,439,503,635,448,399,587,592,663,598,...,665,621,733,546,791,572,587,548,701,539
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
mt-nd3,7111,12690,9805,8066,6143,6600,4815,9508,8672,9020,...,6218,9097,6992,5356,10781,8872,8342,8975,6406,3930
mt-nd4,21825,29177,26462,33494,32804,16978,15866,33967,30352,29477,...,33113,29928,24068,16664,35913,29747,25289,41894,30768,16901
mt-nd4l,2540,3058,2926,3801,2663,1873,1980,3266,3416,3179,...,3703,3160,2709,1830,3963,3257,2984,3175,3554,2027
mt-nd5,12329,19694,17297,17376,22612,10367,9914,23960,17992,18814,...,21511,15814,15392,10895,23150,15632,16334,31330,17930,9719


In [19]:
gse130970 = make_table("/home/yaogilbe/CMSE410/cmse410-final-project/GSE130970_all_sample_salmon_tximport_counts_entrez_gene_ID.csv")
gse130970

,440349.1.X_1,440350.1.X_1,440351.1.X_4,440352.1.X_4,440353.1.X_4,440354.1.X_4,440355.1.X_4,440357.1.X_5,440375.1.X_8,440376.1.X_1,...,440528.1.X_5,440529.1.X_5,440534.1.X_5,440538.1.X_6,440548.1.X_7,449058.1.X_7,449060.1.X_8,449063.1.X_8,449064.1.X_8,449065.1.X_5
entrez_id,,,,,,,,,,,,,,,,,,,,,
1,15968,15623,12255,13328,6906,10556,9997,10990,14610,8831,...,12238,12415,8589,5733,11558,10913,7998,11437,12920,12134
10,1898,1635,1476,1359,847,2195,1546,2677,1094,1251,...,1058,1173,1389,806,1151,1614,1001,1216,1806,1381
100,100,72,67,70,112,47,76,80,57,37,...,34,49,30,22,41,35,38,59,38,28
1000,2969,2997,2547,2625,3644,2485,1216,2240,2569,2756,...,3035,2525,2624,2017,2851,2290,1854,2906,2652,2712
10000,500,398,526,587,907,327,442,408,500,394,...,521,501,296,326,374,405,439,644,704,436
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9990,747,655,491,579,668,477,212,519,496,663,...,648,601,734,549,691,554,433,637,713,567
9991,2211,1761,2105,1730,3392,1700,1052,1839,1679,1464,...,1203,1795,1140,873,1204,1427,1740,1900,1616,1388
9992,107,10,11,27,5,8,31,11,6,10,...,11,16,15,9,24,82,16,13,19,18


In addition, each dataframe must have an accompanying metadata dataframe, containing information about the `group` (species) and `condition` (disease status) of each sample. Therefore, `.csv` files will be imported that contain this information.

In [20]:
gse130970_md = pd.read_csv("GSE130970_Metadata.csv", header=0, index_col=0)
gse167216_md = pd.read_csv("GSE167216_Metadata.csv", header=0, index_col=0)
gse130970_md

,440349.1.X_1,440350.1.X_1,440351.1.X_4,440352.1.X_4,440353.1.X_4,440354.1.X_4,440355.1.X_4,440357.1.X_5,440375.1.X_8,440376.1.X_1,...,440528.1.X_5,440529.1.X_5,440534.1.X_5,440538.1.X_6,440548.1.X_7,449058.1.X_7,449060.1.X_8,449063.1.X_8,449064.1.X_8,449065.1.X_5
condition,5,4,5,5,6,6,5,6,4,4,...,1,1,0,0,0,6,6,5,3,0
group,human,human,human,human,human,human,human,human,human,human,...,human,human,human,human,human,human,human,human,human,human


## Data formatting

For the dataset GSE130970, the gene IDs are instead recorded as `entrez_id`. We want to map these to gene names, so we use the `mygene` package to query the Entrez ID system for their matching gene symbols, creating a new pandas dataframe with these symbols.

In [21]:
def convert_entrez(table, genome):
    
    mg = mygene.MyGeneInfo()
    entrez_ID_list = table.index.tolist()
    
    print("Starting MyGene.info query...")
    
    results = mg.querymany(entrez_ID_list,
                           scopes='entrezgene',
                           fields='symbol',
                           species=genome,
                           as_dataframe=True)
    
    # Drop entries without corresponding gene symbols 
    results = results.dropna(subset=['symbol'])
    mapping = results[['symbol']]
    mapping = mapping[~mapping.index.duplicated(keep='first')]
    
    # Find where mapping index intersects with original Entrez ID table and overwrite - replaces labels
    table = table.loc[table.index.intersection(mapping.index)]
    table.index = mapping.loc[table.index, 'symbol']
    
    # Duplicate gene symbols have their expression patterns averaged
    table = table.groupby(table.index).mean()
    
    print(f"Mapped {len(table)} genes to symbols")

    # Convert index (row names) to strings and all lowercase  
    table = lowercase_index(table)
    
    return table

In [22]:
gse130970_symbols = convert_entrez(gse130970, "human")
gse130970_symbols

Starting MyGene.info query...


159 input query terms found no hit:	['100126582', '100127889', '100128374', '100130285', '100132705', '100133144', '100133301', '1001340


Mapped 19426 genes to symbols


,440349.1.X_1,440350.1.X_1,440351.1.X_4,440352.1.X_4,440353.1.X_4,440354.1.X_4,440355.1.X_4,440357.1.X_5,440375.1.X_8,440376.1.X_1,...,440528.1.X_5,440529.1.X_5,440534.1.X_5,440538.1.X_6,440548.1.X_7,449058.1.X_7,449060.1.X_8,449063.1.X_8,449064.1.X_8,449065.1.X_5
symbol,,,,,,,,,,,,,,,,,,,,,
a1bg,15968.0,15623.0,12255.0,13328.0,6906.0,10556.0,9997.0,10990.0,14610.0,8831.0,...,12238.0,12415.0,8589.0,5733.0,11558.0,10913.0,7998.0,11437.0,12920.0,12134.0
a1cf,21465.0,23403.0,18116.0,17159.0,9293.0,19942.0,10803.0,15732.0,18271.0,19720.0,...,17101.0,17088.0,18674.0,18042.0,19301.0,16481.0,17023.0,21535.0,25753.0,18947.0
a2m,85499.0,70829.0,97974.0,46332.0,79707.0,38391.0,59010.0,20908.0,105181.0,25295.0,...,65044.0,103781.0,43988.0,35546.0,89737.0,39190.0,36627.0,53106.0,52269.0,67573.0
a2ml1,93.0,94.0,88.0,107.0,128.0,56.0,31.0,36.0,92.0,103.0,...,88.0,77.0,66.0,126.0,100.0,82.0,73.0,89.0,122.0,86.0
a3galt2,0.0,0.0,2.0,0.0,0.0,0.0,6.0,0.0,1.0,1.0,...,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
zyg11a,222.0,180.0,317.0,231.0,146.0,177.0,106.0,110.0,140.0,190.0,...,174.0,179.0,344.0,164.0,214.0,188.0,223.0,190.0,239.0,308.0
zyg11b,4064.0,2989.0,2904.0,3374.0,1993.0,2806.0,1431.0,3217.0,3090.0,3477.0,...,2217.0,2713.0,2586.0,2311.0,2320.0,3264.0,2567.0,3913.0,3630.0,2571.0
zyx,812.0,481.0,537.0,527.0,1194.0,619.0,369.0,501.0,292.0,275.0,...,250.0,300.0,141.0,153.0,193.0,203.0,397.0,338.0,265.0,155.0


In [24]:
# Save the completed tables to .csv files, in case they are needed later

gse130970_symbols.to_csv("GSE130970_all_sample_counts_gene_symbol.csv")
gse167216.to_csv("GSE167216_all_sample_counts_gene_symbol.csv")

In [48]:
# Deprecated function - do not use!

""" 
To use the package for differential gene expression analysis, 
we combine the dataframes for GSE130970 and GSE167216, and their 
accompanying metadata. However, the genes between each species do
not entirely overlap. We must therefore write code to sort and 
save the similar genes between the species. 
"""

def common_genes(df1, df2):
    return (df1.index).intersection(df2.index)

def subset_by_genes(df, genes):
    return df.loc[(df.index).intersection(genes)]

"""
# Create new 
    df1_sub = df1.loc[df1.index.intersection(common_genes)]
    df2_sub = df2.loc[df2.index.intersection(common_genes)]
    
    # Align ordering
    df1_sub = df1_sub.sort_index()
    df2_sub = df2_sub.sort_index()

    return df1_sub.index
"""

'\n# Create new \n    df1_sub = df1.loc[df1.index.intersection(common_genes)]\n    df2_sub = df2.loc[df2.index.intersection(common_genes)]\n\n    # Align ordering\n    df1_sub = df1_sub.sort_index()\n    df2_sub = df2_sub.sort_index()\n\n    return df1_sub.index\n'

In [49]:
commons = common_genes(gse130970_symbols, gse167216)
commons

Index(['a1bg', 'a1cf', 'a2m', 'a3galt2', 'a4galt', 'a4gnt', 'aaas', 'aacs',
       'aadac', 'aadacl2',
       ...
       'zwilch', 'zwint', 'zxda', 'zxdb', 'zxdc', 'zyg11a', 'zyg11b', 'zyx',
       'zzef1', 'zzz3'],
      dtype='object', length=15183)

In [51]:
gse130970_symbol_commons = subset_by_genes(gse130970_symbols, commons)
gse130970_symbol_commons

,440349.1.X_1,440350.1.X_1,440351.1.X_4,440352.1.X_4,440353.1.X_4,440354.1.X_4,440355.1.X_4,440357.1.X_5,440375.1.X_8,440376.1.X_1,...,440528.1.X_5,440529.1.X_5,440534.1.X_5,440538.1.X_6,440548.1.X_7,449058.1.X_7,449060.1.X_8,449063.1.X_8,449064.1.X_8,449065.1.X_5
a1bg,15968.0,15623.0,12255.0,13328.0,6906.0,10556.0,9997.0,10990.0,14610.0,8831.0,...,12238.0,12415.0,8589.0,5733.0,11558.0,10913.0,7998.0,11437.0,12920.0,12134.0
a1cf,21465.0,23403.0,18116.0,17159.0,9293.0,19942.0,10803.0,15732.0,18271.0,19720.0,...,17101.0,17088.0,18674.0,18042.0,19301.0,16481.0,17023.0,21535.0,25753.0,18947.0
a2m,85499.0,70829.0,97974.0,46332.0,79707.0,38391.0,59010.0,20908.0,105181.0,25295.0,...,65044.0,103781.0,43988.0,35546.0,89737.0,39190.0,36627.0,53106.0,52269.0,67573.0
a3galt2,0.0,0.0,2.0,0.0,0.0,0.0,6.0,0.0,1.0,1.0,...,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0
a4galt,36.0,26.0,16.0,15.0,48.0,18.0,26.0,14.0,6.0,10.0,...,5.0,18.0,11.0,3.0,13.0,14.0,15.0,11.0,15.0,24.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
zyg11a,222.0,180.0,317.0,231.0,146.0,177.0,106.0,110.0,140.0,190.0,...,174.0,179.0,344.0,164.0,214.0,188.0,223.0,190.0,239.0,308.0
zyg11b,4064.0,2989.0,2904.0,3374.0,1993.0,2806.0,1431.0,3217.0,3090.0,3477.0,...,2217.0,2713.0,2586.0,2311.0,2320.0,3264.0,2567.0,3913.0,3630.0,2571.0
zyx,812.0,481.0,537.0,527.0,1194.0,619.0,369.0,501.0,292.0,275.0,...,250.0,300.0,141.0,153.0,193.0,203.0,397.0,338.0,265.0,155.0
zzef1,2441.0,1990.0,2057.0,2134.0,2248.0,1562.0,1205.0,1396.0,1741.0,1803.0,...,2099.0,2053.0,2191.0,1532.0,2255.0,2231.0,2276.0,2433.0,2669.0,2027.0


In [52]:
gse167216_commons = subset_by_genes(gse167216, commons)
gse167216_commons

,I00001,I00002,I00003,I00004,I00005,I00006,I00007,I00008,I00009,I00010,...,I00027,I00028,I00029,I00030,I00031,I00032,I00033,I00034,I00035,I00036
a1bg,166,0,2,5,0,2,57,4,2,1,...,0,1,1,18,2,2,3,2,2,23
a1cf,1298,1713,1498,1891,2637,1337,1396,2199,1940,1896,...,2392,2000,2561,806,1238,2019,1866,3028,2869,901
a2m,7,4,11,6,3,125,2281,7,6,5,...,4,2,10,22576,54,19,8,6,7,2962
a3galt2,0,0,1,0,4,0,4,2,0,0,...,1,0,0,0,7,0,2,0,1,0
a4galt,153,19,22,15,23,8,83,32,20,20,...,28,18,25,19,361,15,25,24,27,9
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
mt-nd3,7111,12690,9805,8066,6143,6600,4815,9508,8672,9020,...,6218,9097,6992,5356,10781,8872,8342,8975,6406,3930
mt-nd4,21825,29177,26462,33494,32804,16978,15866,33967,30352,29477,...,33113,29928,24068,16664,35913,29747,25289,41894,30768,16901
mt-nd4l,2540,3058,2926,3801,2663,1873,1980,3266,3416,3179,...,3703,3160,2709,1830,3963,3257,2984,3175,3554,2027
mt-nd5,12329,19694,17297,17376,22612,10367,9914,23960,17992,18814,...,21511,15814,15392,10895,23150,15632,16334,31330,17930,9719


In [ ]:
gse130970_symbols_.to_csv("GSE130970_all_sample_counts_gene_symbol.csv")
gse167216.to_csv("GSE167216_all_sample_counts_gene_symbol.csv")

Code to create data subsets - define at least one disease/treatment group and the control group 

In [25]:
# Create function to make a subset of dataset based on user-specified metadata "condition" value list

def subset_by_condition(counts_df, metadata, condition_name, conditions_to_keep, counts_are_transposed=False):
    """
    Subset RNA-seq dataset based on selected condition values.

    Parameters:
        counts_df (pd.DataFrame): 
            - genes x samples (default) OR samples x genes (if transposed)
        metadata (pd.DataFrame): samples x metadata
        condition_name (str): column in metadata (e.g., "condition")
        conditions_to_keep (list): list of condition values to retain
        counts_are_transposed (bool): 
            False = counts are genes x samples (default)
            True = counts are samples x genes

    Returns:
        subset_counts (pd.DataFrame)
        subset_metadata (pd.DataFrame)
    """

    # Ensure condition column exists
    if condition_name not in metadata.columns:
        raise ValueError(f"{condition_name} not found in metadata columns")

    # Filter metadata
    subset_metadata = metadata[metadata[condition_name].isin(conditions_to_keep)].copy()

    if subset_metadata.empty:
        raise ValueError("No samples match the specified conditions")

    # Get sample IDs
    sample_ids = subset_metadata.index

    # Subset counts
    if counts_are_transposed:
        # samples x genes
        subset_counts = counts_df.loc[sample_ids]
    else:
        # genes x samples
        subset_counts = counts_df[sample_ids]

    # Ensure alignment
    subset_metadata = subset_metadata.loc[sample_ids]

    print(f"✅ Subset complete:")
    print(f"  Samples retained: {len(sample_ids)}")
    print(f"  Conditions: {conditions_to_keep}")
    print(f"  Counts shape: {subset_counts.shape}")

    return subset_counts, subset_metadata

In [26]:
gse130970_sub_06, gse130970_sub_06_md = subset_by_condition(gse130970_symbols, gse130970_md.T, "condition", ["0","6"], counts_are_transposed=False)

✅ Subset complete:
  Samples retained: 12
  Conditions: ['0', '6']
  Counts shape: (19426, 12)


In [29]:
gse130970_sub_06_t100 = gse130970_sub_06.head(100).copy()

In [30]:
gse130970_sub_06_t100

,440353.1.X_4,440354.1.X_4,440357.1.X_5,440422.1.X_1,440492.1.X_5,440527.1.X_4,440534.1.X_5,440538.1.X_6,440548.1.X_7,449058.1.X_7,449060.1.X_8,449065.1.X_5
symbol,,,,,,,,,,,,
a1bg,6906.0,10556.0,10990.0,10582.0,12276.0,11563.0,8589.0,5733.0,11558.0,10913.0,7998.0,12134.0
a1cf,9293.0,19942.0,15732.0,19612.0,18474.0,18603.0,18674.0,18042.0,19301.0,16481.0,17023.0,18947.0
a2m,79707.0,38391.0,20908.0,39288.0,98866.0,86713.0,43988.0,35546.0,89737.0,39190.0,36627.0,67573.0
a2ml1,128.0,56.0,36.0,93.0,89.0,84.0,66.0,126.0,100.0,82.0,73.0,86.0
a3galt2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...
abhd17c,128.0,62.0,52.0,62.0,45.0,68.0,22.0,21.0,26.0,63.0,40.0,27.0
abhd18,734.0,827.0,951.0,1085.0,927.0,819.0,1007.0,780.0,875.0,916.0,729.0,925.0
abhd2,16847.0,18029.0,14277.0,20068.0,18424.0,20395.0,16215.0,7856.0,11243.0,19539.0,17771.0,12401.0


## Differential gene expression analysis

Now, we perform **differential gene expression analysis**. In the original paper, this was done in R using the `limma` package; to perform this in a Python environment, we can instead use the `PyDESeq2` package and pipeline. This starts with importing the `pydeseq2` package and its associated functions.
```bash
pip install pydeseq2
```

In [31]:
# Import pydeseq2 important packages

import pickle as pkl

from pydeseq2.dds import DeseqDataSet
from pydeseq2.default_inference import DefaultInference
from pydeseq2.ds import DeseqStats
from pydeseq2.utils import load_example_data # not necessary, used for tests

However, the PyDESeq2 package can only perform pairwise comparisons, but there might be multiple different disease or treatment types or states. Therefore, we must write code for Python that will automatically perform pairwise DESeq2 analysis between all conditions against a user-specified control variable.

In [33]:
# Deprecated function for deseq2
def dep_run_deseq2(counts_df, metadata, condition_name, condition_label, control_label):

    # Detect valid control label - terminates program early if not valid
    if control_label not in metadata[condition_name].unique():
        raise ValueError(f"Designated control label cannot be found in condition types")

    # Default variable for deseq2 process
    inference = DefaultInference(n_cpus=8)

    # Create a dds object for the analysis - code modified from pydeseq2 documentation 
    # https://pydeseq2.readthedocs.io/en/latest/auto_examples/plot_minimal_pydeseq2_pipeline.html#data-loading
    dds = DeseqDataSet(
        counts=counts_df,
        metadata=metadata,
        design="~condition",
        refit_cooks=True,
        inference=inference, # n_cpus=8, # n_cpus can be specified here or in the inference object
    )

    # Run DESeq2 pipeline
    dds.deseq2() # the magic!
    
    stat_res = DeseqStats(dds,contrast=(condition_name, condition_label, control_label))
    stat_res.summary()

    res_df = stat_res.results_df.copy()
    results = res_df # append new creation to results dictionary

    return results

In [34]:
gse130970_test_results = dep_run_deseq2(gse130970_sub_06_t100.T, gse130970_sub_06_md, condition_name="condition", condition_label="6", control_label="0")

Using None as control genes, passed at DeseqDataSet initialization


Fitting size factors...
... done in 0.00 seconds.

Fitting dispersions...
... done in 0.04 seconds.

Fitting dispersion trend curve...
... done in 0.02 seconds.

Fitting MAP dispersions...
... done in 0.05 seconds.

Fitting LFCs...
... done in 0.07 seconds.

Calculating cook's distance...
... done in 0.01 seconds.

Replacing 0 outlier genes.

Running Wald tests...


Log2 fold change & Wald test p-value: condition 6 vs 0
             baseMean  log2FoldChange     lfcSE      stat    pvalue      padj
symbol                                                                       
a1bg      9689.851767       -0.084057  0.183406 -0.458312  0.646728  0.771399
a1cf     17406.927073       -0.406347  0.207360 -1.959620  0.050040  0.163777
a2m      53746.037742       -0.341922  0.315911 -1.082337  0.279103  0.477165
a2ml1       84.876943       -0.580696  0.297931 -1.949097  0.051284  0.163777
a3galt2      0.163303       -1.661856  3.443044 -0.482670  0.629330  0.762061
...               ...             ...       ...       ...       ...       ...
abhd17c     48.274602        1.183240  0.300781  3.933890  0.000084  0.001655
abhd18     868.939233       -0.280667  0.158507 -1.770692  0.076612  0.216702
abhd2    15434.484205        0.391122  0.129252  3.026033  0.002478  0.022301
abhd3      531.284966        0.262102  0.196391  1.334596  0.182009  0.391281
abhd4    

... done in 21.61 seconds.

